In [ ]:
import TechAna_DRAFT as TechAna
import pandas as pd
import requests
import numpy as np
import importlib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
INDUSTRY_MAP = {
    1: {'name': 'Banks','module': 'TotalScore_Bank'},
    2: {'name': 'Consumer','module': 'TotalScore_Consumer'},
    3: {'name': 'Financials','module': 'TotalScore_Financials'},
    4: {'name': 'Construction_and_materials','module': 'TotalScore_CM'},
    5: {'name': 'Goods_and_services','module': 'TotalScore_GS'},
    6: {'name': 'HealthCare','module': 'TotalScore_HealthCare'},
    7: {'name': 'Insurance','module': 'TotalScore_Insurance'},
    8: {'name': 'Materials','module': 'TotalScore_Materials'},
    9: {'name': 'RealEstate','module': 'TotalScore_RealEstate'},
    10: {'name': 'Utilities_and_Energy','module': 'TotalScore_UtiEne'},
    11: {'name': 'TechTele','module': 'TotalScore_TechTele'},
}

In [ ]:
def _parse_stock_payload(payload):
    records = []
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        for sym, rows in payload.items():
            if isinstance(rows, list):
                for r in rows:
                    if 'symbol' not in r:
                        r = {**r, 'symbol': sym}
                    records.append(r)
            elif isinstance(rows, dict):
                if 'symbol' not in rows:
                    rows = {**rows, 'symbol': sym}
                records.append(rows)

    df = pd.DataFrame(records)
    if df.empty:
        return df

    for col in ['date', 'Date', 'trading_date', 'TradingDate']:
        if col in df.columns:
            df['date'] = pd.to_datetime(df[col], errors='coerce')
            break

    # Use Adj Close for all OHLC if available
    if 'adj_close' in df.columns:
        adj = pd.to_numeric(df['adj_close'], errors='coerce')
        df['open'] = adj
        df['high'] = adj
        df['low'] = adj
        df['close'] = adj

    else:
        for col in ['open', 'Open']:
            if col in df.columns:
                df['open'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['high', 'High']:
            if col in df.columns:
                df['high'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['low', 'Low']:
            if col in df.columns:
                df['low'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['close', 'Close']:
            if col in df.columns:
                df['close'] = pd.to_numeric(df[col], errors='coerce')
                break

    return df.dropna(subset=['symbol', 'date'])


def _compute_max_drawdown(series):
    if series is None or len(series) == 0:
        return 0.0
    running_max = series.cummax()
    drawdown = (series - running_max) / running_max
    return float(drawdown.min()) if len(drawdown) > 0 else 0.0


def _fetch_prices(symbols, start_date, end_date):
    if not symbols:
        return pd.DataFrame()
    params = {
        "symbols": ",".join(symbols),
        "start_date": start_date,
        "end_date": end_date
    }
    resp = requests.get(
        "http://192.168.8.190:8000/MKD/stock_daily",
        params=params,
        headers={"accept": "application/json"},
        timeout=30
    )
    resp.raise_for_status()
    payload = resp.json()
    return _parse_stock_payload(payload)


def _run_quarter_trades(symbols, start_date, end_date, df_prices, entry_override=None, original_entry_override=None):
    entry_override = entry_override or {}
    original_entry_override = original_entry_override or {}

    trades = []
    for sym in symbols:
        df_sym = df_prices[df_prices['symbol'] == sym].sort_values('date').reset_index(drop=True)
        if df_sym.empty:
            continue

        entry_row = df_sym.iloc[0]
        entry_date = entry_row['date']
        entry_price = entry_override.get(sym, entry_row.get('open', np.nan))
        if pd.isna(entry_price):
            continue

        original_entry_price = original_entry_override.get(sym, entry_price)

        sl_price = entry_price * 0.85
        tp_price = entry_price * 1.25

        exit_date = df_sym.iloc[-1]['date']
        exit_price = df_sym.iloc[-1]['close'] if 'close' in df_sym.columns else entry_price
        exit_reason = 'Keep Position'

        start_idx = 3 if len(df_sym) > 3 else len(df_sym)
        for i in range(start_idx, len(df_sym)):
            row = df_sym.iloc[i]
            day_open = row.get('open', np.nan)
            day_low = row.get('low', np.nan)
            day_high = row.get('high', np.nan)

            # Stoploss check (-15%)
            if pd.notna(day_low) and day_low <= sl_price:
                if pd.notna(day_open) and day_open <= sl_price:
                    exit_price = day_open
                else:
                    exit_price = sl_price
                exit_date = row['date']
                exit_reason = 'Stop Loss'
                break

            # Take profit check (+25%)
            if pd.notna(day_high) and day_high >= tp_price:
                exit_price = tp_price
                exit_date = row['date']
                exit_reason = 'Take Profit'
                break

        ret_pct = (exit_price - entry_price) / entry_price if entry_price else 0.0
        cum_ret_pct = (exit_price - original_entry_price) / original_entry_price if original_entry_price else 0.0

        max_dd = 0.0
        max_ret = 0.0
        min_ret = 0.0
        if 'close' in df_sym.columns:
            hold_df = df_sym[(df_sym['date'] >= entry_date) & (df_sym['date'] <= exit_date)]
            close_series = hold_df['close'].dropna()
            price_path = pd.concat([pd.Series([entry_price]), close_series], ignore_index=True)
            max_dd = _compute_max_drawdown(price_path)

            if not close_series.empty and entry_price:
                max_ret = (close_series.max() - entry_price) / entry_price
                min_ret = (close_series.min() - entry_price) / entry_price

        trades.append({
            'Symbol': sym,
            'Entry_Date': entry_date,
            'Entry_Price': entry_price,
            'Original_Entry_Price': original_entry_price,
            'Exit_Date': exit_date,
            'Exit_Price': exit_price,
            'Exit_Reason': exit_reason,
            'Return_Pct': ret_pct,
            'Max_Return_Pct': max_ret,
            'Min_Return_Pct': min_ret,
            'Cum_Return_Pct': cum_ret_pct,
            'Max_Drawdown': max_dd
        })

    return pd.DataFrame(trades)

def print_summary(df_trades, label, start_date, end_date, industry_name):
    if df_trades.empty:
        print(f'No trades generated for {label}.')
        return

    win_rate = (df_trades['Return_Pct'] > 0).mean()
    avg_returns = df_trades['Return_Pct'].mean()
    max_drawdown = df_trades['Max_Drawdown'].min()
    max_return = df_trades['Return_Pct'].max()
    min_return = df_trades['Return_Pct'].min()

    summary_df = pd.DataFrame([{
        'Industry': industry_name,
        'Label': label,
        'Period_Start': start_date,
        'Period_End': end_date,
        'Win_Rate': win_rate,
        'Average_Return': avg_returns,
        'Max_Return': max_return,
        'Min_Return': min_return,
        'Max_Drawdown': max_drawdown,
        'Deals': len(df_trades)
    }])

    print(f'\nTrade Results ({label}):')
    print(df_trades.to_string(index=False))
    print('\nSummary:')
    print(summary_df.to_string(index=False))

def run_backtest(industry_id):
    if industry_id not in INDUSTRY_MAP:
        print(f"Error: Industry ID {industry_id} not found in configuration.")
        return

    config = INDUSTRY_MAP[industry_id]
    industry_name = config['name']
    module_name = config['module']
    
    print(f"STARTING BACKTEST FOR: {industry_id} - {industry_name} (Module: {module_name})")
    try:
        TotalScore_Module = importlib.import_module(module_name)
    except ImportError:
        print(f"Error: Could not import module '{module_name}'. Check if file exists.")
        return

    TECH_START_DATE = TechAna.START_DATE
    TECH_END_DATE = TechAna.END_DATE

    tech_start = pd.to_datetime(TECH_START_DATE)
    tech_end = pd.to_datetime(TECH_END_DATE)
    tech_q = tech_end.to_period('Q')

    curr_q = tech_q + 1
    next_q = curr_q + 1

    PREV_START_DATE = tech_q.start_time.strftime('%Y-%m-%d')
    PREV_END_DATE = tech_q.end_time.strftime('%Y-%m-%d')
    START_DATE = curr_q.start_time.strftime('%Y-%m-%d')
    END_DATE = curr_q.end_time.strftime('%Y-%m-%d')
    NEXT_START_DATE = next_q.start_time.strftime('%Y-%m-%d')
    NEXT_END_DATE = next_q.end_time.strftime('%Y-%m-%d')

# Ranking for quarter t-1 (entry list for backtest quarter t)
    df_total_prev = TotalScore_Module.get_total_score(PREV_START_DATE, PREV_END_DATE, industry=industry_name)
    df_rank_prev = df_total_prev[['Symbol', 'Final_Score']].copy()
    df_rank_prev = df_rank_prev.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_PREV = df_rank_prev.head(10)['Symbol'].tolist()

# Ranking for quarter t (used for rollover filter and next-quarter new entries)
    df_total_curr = TotalScore_Module.get_total_score(START_DATE, END_DATE, industry=industry_name)
    df_rank_curr = df_total_curr[['Symbol', 'Final_Score']].copy()
    df_rank_curr = df_rank_curr.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_CURR = df_rank_curr.head(10)['Symbol'].tolist()
    TOP13_CURR = df_rank_curr.head(13)['Symbol'].tolist()

    df_prices_curr = _fetch_prices(TOP10_PREV, START_DATE, END_DATE)
    df_trades_curr = _run_quarter_trades(TOP10_PREV, START_DATE, END_DATE, df_prices_curr)

    rollover_symbols = []
    entry_override_next = {}
    original_entry_override_next = {}

    if not df_trades_curr.empty:
        pending_df = df_trades_curr[df_trades_curr['Exit_Reason'] == 'Keep Position']
        for _, row in pending_df.iterrows():
            sym = row['Symbol']
            if sym in TOP13_CURR:
                rollover_symbols.append(sym)
                original_entry_price = row.get('Original_Entry_Price', row['Entry_Price'])
                if row.get('Return_Pct', 0) > 0:
                    reference_price = row['Exit_Price']
                else:
                    reference_price = original_entry_price

                entry_override_next[sym] = reference_price
                original_entry_override_next[sym] = original_entry_price
                df_trades_curr.loc[df_trades_curr['Symbol'] == sym, 'Exit_Reason'] = 'Rollover'

    print_summary(df_trades_curr, f"{curr_q.year}Q{curr_q.quarter}", START_DATE, END_DATE, industry_name)

    symbols_next = TOP10_CURR + [s for s in rollover_symbols if s not in TOP10_CURR]
    df_prices_next = _fetch_prices(symbols_next, NEXT_START_DATE, NEXT_END_DATE)
    df_trades_next = _run_quarter_trades(
        symbols_next,
        NEXT_START_DATE,
        NEXT_END_DATE,
        df_prices_next,
        entry_override=entry_override_next,
        original_entry_override=original_entry_override_next
    )

    print_summary(df_trades_next, f"{next_q.year}Q{next_q.quarter}", NEXT_START_DATE, NEXT_END_DATE, industry_name)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 7 - Insurance (Module: TotalScore_Insurance)
Success
[Insurance] Mega Caps: ['BVH', 'PVI', 'BIC', 'VNR']
[Insurance] Large Caps: ['MIG', 'PTI', 'BMI', 'PRE', 'PGI', 'ABI']
[Insurance] Mega Caps: ['BVH', 'PVI', 'BIC', 'VNR']
[Insurance] Large Caps: ['MIG', 'PTI', 'BMI', 'PRE', 'PGI', 'ABI']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   MIG 2023-01-03     11731.19              11731.19 2023-03-31    12148.80    Rollover    0.035598        0.074433       -0.103560        0.035598     -0.165662
   PRE 2023-01-03     13414.42              13414.42 2023-03-31    13581.30    Rollover    0.012440        0.132948       -0.040462        0.012440     -0.153061
   BVH 2023-01-03     44618.58              44618.58 2023-03-31    45320.13    Rollover    0.015723        0.073375        0.000000        0.015723     -0.062500
   PVI 2023-0

In [ ]:
@"STARTING BACKTEST FOR: 1 - Banks (Module: TotalScore_Bank)

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   CTG 2023-01-03     17189.20              17189.20 2023-03-31    17925.88      Rollover    0.042857        0.110714       -0.021429        0.042857     -0.118971
   TCB 2023-01-03     12967.38              12967.38 2023-03-31    13392.54      Rollover    0.032787        0.071038       -0.045537        0.032787     -0.108844
   VCB 2023-01-03     46462.50              46462.50 2023-03-31    51412.50      Rollover    0.106538        0.162228        0.000000        0.106538     -0.113542
   STB 2023-01-03     23500.00              23500.00 2023-03-31    26200.00      Rollover    0.114894        0.153191       -0.008511        0.114894     -0.138376
   BID 2023-01-03     29886.48              29886.48 2023-03-31    33513.48      Rollover    0.121359        0.165049       -0.010922        0.121359     -0.068553
   TPB 2023-01-03     10231.68              10231.68 2023-03-31    11508.48      Rollover    0.124789        0.155251        0.000000        0.124789     -0.080000
   MBB 2023-01-03      9734.40               9734.40 2023-03-31     9869.60      Rollover    0.013889        0.094444       -0.047222        0.013889     -0.129442
   ACB 2023-01-03     13299.93              13299.93 2023-03-31    14647.50      Rollover    0.101322        0.160793       -0.002202        0.101322     -0.094877
   SHB 2023-01-03      6485.44               6485.44 2023-03-31     6703.70 Keep Position    0.033654        0.076923       -0.059615        0.033654     -0.126785
   LPB 2023-01-03      8438.43               8438.43 2023-03-31     9520.28      Rollover    0.128205        0.135531       -0.010989        0.128205     -0.096667

Summary:
Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
   Banks 2023Q1   2023-01-01 2023-03-31       1.0        0.082029    0.128205    0.013889     -0.138376     10

Trade Results (2023Q2):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VCB 2023-04-03     51412.50              46462.50 2023-06-30    56250.00 Keep Position    0.094092        0.148796       -0.042670        0.210654     -0.057112
   BID 2023-04-03     33513.48              29886.48 2023-06-30    31446.09 Keep Position   -0.061688       -0.004329       -0.062771        0.052184     -0.062771
   TPB 2023-04-03     11508.48              10231.68 2023-06-30    12983.40 Keep Position    0.128159        0.184685        0.006757        0.268941     -0.047713
   CTG 2023-04-03     17925.88              17189.20 2023-06-30    18110.05 Keep Position    0.010274        0.027397       -0.058219        0.053571     -0.083333
   BAB 2023-04-03     11086.92              11086.92 2023-06-30    11247.60 Keep Position    0.014493        0.050725       -0.036232        0.014493     -0.050000
   STB 2023-04-03     26200.00              23500.00 2023-06-30    29800.00 Keep Position    0.137405        0.156489       -0.047710        0.268085     -0.072491
   HDB 2023-04-03      9866.57               9866.57 2023-06-30     9997.50 Keep Position    0.013270        0.045956       -0.051680        0.013270     -0.068527
   TCB 2023-04-03     13392.54              12967.38 2023-06-30    15282.14 Keep Position    0.141093        0.174603        0.012346        0.178506     -0.065147
   VPB 2023-04-03     18679.32              18679.32 2023-06-30    17489.84 Keep Position   -0.063679        0.009434       -0.096698       -0.063679     -0.105140
   MBB 2023-04-03      9869.60               9734.40 2023-06-30    11200.90 Keep Position    0.134889        0.162980       -0.010959        0.150651     -0.039894
   ACB 2023-04-03     14647.50              13299.93 2023-06-30    15479.10 Keep Position    0.056774        0.068756       -0.034000        0.163848     -0.045454
   LPB 2023-04-03      9520.28               8438.43 2023-06-30     9365.73 Keep Position   -0.016234        0.022727       -0.149351        0.109890     -0.168254

Summary:
Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
   Banks 2023Q2   2023-04-01 2023-06-30      0.75        0.049071    0.141093   -0.063679     -0.168254     12"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-5-ddee2912d6f4>, line 1)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 11 - TechTele (Module: TotalScore_TechTele)
Success
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['SGT', 'VEC', 'ICT', 'POT', 'UNI', 'PIA']
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['SGT', 'VEC', 'ICT', 'POT', 'UNI', 'PIA']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   FPT 2023-01-03     49760.00              49760.00 2023-03-31    49200.20      Rollover   -0.011250        0.050000       -0.030000       -0.011250     -0.076190
   PIA 2023-01-03     21270.60              21270.60 2023-03-31    20452.50      Rollover   -0.038462        0.126923       -0.115385       -0.038462     -0.197952
   CKV 2023-01-03     15791.60              15791.60 2023-01-27    12861.20     Stop Loss   -0.185567        0.000000       -0.185567       -0.185567     -0.185567
   UNI 202

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 2 - Consumer (Module: TotalScore_Consumer)
Success
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'SBT']
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'SBT']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   MSH 2023-01-03     17062.50              17062.50 2023-03-31   17325.000      Rollover    0.015385        0.080000       -0.023077        0.015385     -0.076923
   MSN 2023-01-03     96000.00              96000.00 2023-02-27   79900.000     Stop Loss   -0.167708        0.080208       -0.167708       -0.167708     -0.229508
   MCH 2023-01-03     34046.50              34046.50 2023-03-31   30024.900 Keep Position   -0.118121        0.013423       -0.131544       -0.118121     -0.143046
   VNM 2023

In [ ]:
@"Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   IMP 2023-01-03     25121.46              25121.46 2023-02-22    21324.03     Stop Loss   -0.151163        0.000000       -0.151163       -0.151163     -0.151163
   VDP 2023-01-03     26265.00              26265.00 2023-03-31    25916.80      Rollover   -0.013257        0.094841       -0.058667       -0.013257     -0.098734
   TRA 2023-01-03     82544.02              82544.02 2023-03-31    78195.02 Keep Position   -0.052687        0.000000       -0.090622       -0.052687     -0.090622
   DHT 2023-01-03     12083.65              12083.65 2023-03-31    12525.35      Rollover    0.036554        0.083551       -0.033943        0.036554     -0.055422
   TNH 2023-01-03     15169.32              15169.32 2023-03-31    15997.53 Keep Position    0.054598        0.114943       -0.008621        0.054598     -0.072165
   AGP 2023-01-03     14999.60              14999.60 2023-03-02    18749.50   Take Profit    0.250000        0.259091        0.000000        0.250000     -0.043478
   DAN 2023-01-03     31534.65              31534.65 2023-03-06    25591.95     Stop Loss   -0.188450        0.000000       -0.188450       -0.188450     -0.188450
   UPH 2023-01-03     15800.00              15800.00 2023-01-12    11500.00     Stop Loss   -0.272152        0.000000       -0.272152       -0.272152     -0.272152
   DHG 2023-01-03     71383.68              71383.68 2023-03-31    76588.74      Rollover    0.072917        0.145833       -0.002315        0.072917     -0.064646
   PPP 2023-01-03     10303.92              10303.92 2023-03-31    11409.04 Keep Position    0.107252        0.128788       -0.007576        0.107252     -0.093960

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
HealthCare 2023Q1   2023-01-01 2023-03-31       0.5       -0.015639        0.25   -0.272152     -0.272152     10

Trade Results (2023Q2):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DHG 2023-04-03     76588.74              71383.68 2023-05-05  95735.9250   Take Profit    0.250000        0.251348       -0.012945        0.341146     -0.016060
   DBD 2023-04-03     29930.32              29930.32 2023-06-12  37412.9000   Take Profit    0.250000        0.275773        0.000000        0.250000     -0.043573
   VDP 2023-04-03     26265.00              26265.00 2023-06-30  28464.8000 Keep Position    0.083754        0.150276       -0.027116        0.083754     -0.073879
   DMC 2023-04-03     38401.60              38401.60 2023-05-29  48002.0000   Take Profit    0.250000        0.289872        0.000000        0.250000     -0.029345
   MED 2023-04-03     19778.00              19778.00 2023-06-30  22475.0000 Keep Position    0.136364        0.181818        0.000000        0.136364     -0.142308
   DHT 2023-04-03     12525.35              12083.65 2023-05-31  15656.6875   Take Profit    0.250000        0.259446       -0.017632        0.295692     -0.058455
   MKV 2023-04-03     12752.85              12752.85 2023-06-05  10100.0000     Stop Loss   -0.208020        0.000000       -0.208020       -0.208020     -0.208020
   DNM 2023-04-03     18900.00              18900.00 2023-04-26  16000.0000     Stop Loss   -0.153439        0.126984       -0.153439       -0.153439     -0.248826
   DP3 2023-04-03     34675.00              34675.00 2023-06-02  43343.7500   Take Profit    0.250000        0.284211        0.000000        0.250000     -0.109982
   MKP 2023-04-03     25986.80              25986.80 2023-06-30  26813.1000 Keep Position    0.031797        0.076335       -0.071429        0.031797     -0.071429

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
HealthCare 2023Q2   2023-04-01 2023-06-30       0.8        0.114046        0.25    -0.20802     -0.248826     10"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-8-18792c57e840>, line 1)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 4 - Construction_and_materials (Module: TotalScore_CM)
Success
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'SNZ']
[Construction_and_materials] Large Caps: ['LGC', 'VCG', 'CTR', 'NTP', 'PC1', 'SJG']
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'SNZ']
[Construction_and_materials] Large Caps: ['LGC', 'VCG', 'CTR', 'NTP', 'PC1', 'SJG']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   LBM 2023-01-03     16236.78              16236.78 2023-03-31    17502.72      Rollover    0.077967        0.077967       -0.017413        0.077967     -0.058411
   GMX 2023-01-03     14667.90              14667.90 2023-03-31    15194.44 Keep Position    0.035897        0.164103       -0.046154        0.035897     -0.118943
   VCG 2023-01-03     13376.25              13376.25 2023-03-31    14753.75      Rollover  

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 10 - Utilities_and_Energy (Module: TotalScore_UtiEne)
Success
[Utilities_and_Energy] Mega Caps: ['GAS', 'PLX', 'REE', 'PGV']
[Utilities_and_Energy] Large Caps: ['PVS', 'PVD', 'VSH', 'BWE', 'TOS', 'NT2']
[Utilities_and_Energy] Mega Caps: ['GAS', 'PLX', 'REE', 'PGV']
[Utilities_and_Energy] Large Caps: ['PVS', 'PVD', 'VSH', 'BWE', 'TOS', 'NT2']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VPD 2023-01-03     19583.31              19583.31 2023-03-31    21979.58 Keep Position    0.122363        0.210970       -0.012658        0.122363     -0.121951
   PMG 2023-01-03     11200.00              11200.00 2023-03-31    10000.00 Keep Position   -0.107143        0.062500       -0.133929       -0.107143     -0.184874
   DRL 2023-01-03     50724.24              50724.24 2023-03-31    50879.36 Keep Position    0.003058        0.07339

In [ ]:
@"Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   NLG 2023-01-03     28400.36              28400.36 2023-01-12    23733.63     Stop Loss   -0.164319        0.000000       -0.164319       -0.164319     -0.164319
   KBC 2023-01-03     24700.00              24700.00 2023-03-31    24250.00      Rollover   -0.018219        0.091093       -0.145749       -0.018219     -0.217069
   FIR 2023-01-03     29986.75              29986.75 2023-03-31    32286.54      Rollover    0.076694        0.107023       -0.027368        0.076694     -0.057078
   VRE 2023-01-03     28100.00              28100.00 2023-03-31    29550.00      Rollover    0.051601        0.078292       -0.083630        0.051601     -0.150165
   KDH 2023-01-03     21028.00              21028.00 2023-03-31    20727.60      Rollover   -0.014286        0.010714       -0.130357       -0.014286     -0.139576
   VHM 2023-01-03     49400.00              49400.00 2023-02-24    41000.00     Stop Loss   -0.170040        0.078947       -0.170040       -0.170040     -0.230769
   BCM 2023-01-03     81622.80              81622.80 2023-03-31    80553.93 Keep Position   -0.013095        0.023810       -0.029762       -0.013095     -0.052326
   VIC 2023-01-03     28400.00              28400.00 2023-03-31    27500.00 Keep Position   -0.031690        0.042254       -0.075704       -0.031690     -0.113176
   HDG 2023-01-03     21258.27              21258.27 2023-03-31    19795.55 Keep Position   -0.068807        0.045872       -0.143731       -0.068807     -0.181287
   TIP 2023-01-03     14064.16              14064.16 2023-03-31    13904.34 Keep Position   -0.011364        0.028409       -0.079545       -0.011364     -0.104972

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
RealEstate 2023Q1   2023-01-01 2023-03-31       0.2       -0.036353    0.076694    -0.17004     -0.230769     10

Trade Results (2023Q2):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VHM 2023-04-03     52600.00              52600.00 2023-06-30    55000.00 Keep Position    0.045627        0.083650       -0.096958        0.045627     -0.096958
   VRE 2023-04-03     29550.00              28100.00 2023-06-30    26800.00 Keep Position   -0.093063        0.001692       -0.103215       -0.046263     -0.104730
   D2D 2023-04-03     14824.94              14824.94 2023-06-30    18186.06 Keep Position    0.226721        0.226721        0.000000        0.226721     -0.065056
   KBC 2023-04-03     24700.00              24700.00 2023-06-30    29350.00 Keep Position    0.188259        0.226721       -0.016194        0.188259     -0.083019
   HDC 2023-04-03     19174.83              19174.83 2023-06-30    20283.20 Keep Position    0.057803        0.142805       -0.033233        0.057803     -0.074380
   KDH 2023-04-03     21028.00              21028.00 2023-06-30    23130.80 Keep Position    0.100000        0.130357        0.001786        0.100000     -0.064039
   SZC 2023-04-03     19975.38              19975.38 2023-06-30    23407.13 Keep Position    0.171799        0.213938       -0.024311        0.171799     -0.083582
   IDC 2023-04-03     28685.43              28685.43 2023-06-30    30659.44 Keep Position    0.068816        0.114190       -0.049661        0.068816     -0.054198
   NVL 2023-04-03     12850.00              12850.00 2023-06-30    14850.00 Keep Position    0.155642        0.214008       -0.007782        0.155642     -0.139535
   CEO 2023-04-03     16237.66              16237.66 2023-06-30    16169.72 Keep Position   -0.004184        0.154812       -0.037657       -0.004184     -0.137681
   FIR 2023-04-03     32286.54              29986.75 2023-06-30    28285.45 Keep Position   -0.123924        0.028169       -0.143643       -0.056735     -0.167105

Summary:
  Industry  Label Period_Start Period_End  Win_Rate  Average_Return  Max_Return  Min_Return  Max_Drawdown  Deals
RealEstate 2023Q2   2023-04-01 2023-06-30  0.727273        0.072136    0.226721   -0.123924     -0.167105     11"@

SyntaxError: unterminated string literal (detected at line 1) (<ipython-input-11-6b5cf95ca027>, line 1)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 8 - Materials (Module: TotalScore_Materials)
Success
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'VIF']
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'VIF']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   HPG 2023-01-03     14570.33              14570.33 2023-03-31  15743.5200      Rollover    0.080519        0.148052        0.000000        0.080519     -0.106335
   PHR 2023-01-03     36576.10              36576.10 2023-03-31  35817.8200      Rollover   -0.020732        0.115854       -0.053659       -0.020732     -0.151913
   BFC 2023-01-03     13793.22              13793.22 2023-03-31  13199.4800 Keep Position   -0.043046        0.035488       -0.063407       -0.043046     -0.095506
   NH

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 3 - Financials (Module: TotalScore_Financials)
Success
[Financials] Mega Caps: ['SSI', 'VND', 'VCI', 'FTS']
[Financials] Large Caps: ['BSI', 'TIN', 'DSC', 'TVS', 'BVS', 'VFS']
[Financials] Mega Caps: ['SSI', 'VND', 'VCI', 'FTS']
[Financials] Large Caps: ['BSI', 'TIN', 'DSC', 'TVS', 'BVS', 'VFS']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   SSI 2023-01-03     12628.98              12628.98 2023-03-31  14366.3000    Rollover    0.137566        0.142857       -0.034392        0.137566     -0.155093
   VND 2023-01-03     11845.44              11845.44 2023-03-31  12750.3000    Rollover    0.076389        0.163194       -0.069444        0.076389     -0.200000
   VCI 2023-01-03     18262.27              18262.27 2023-03-23  22827.8375 Take Profit    0.250000        0.263581        0.000000        0.250000     -0.166667
   FTS 

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 5 - Goods_and_services (Module: TotalScore_GS)
Success
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']

Trade Results (2023Q1):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   GEX 2023-01-03     12193.98              12193.98 2023-03-31    11411.72 Keep Position   -0.064151        0.109434       -0.132076       -0.064151     -0.217687
   PVT 2023-01-03     15115.52              15115.52 2023-02-02    12821.20     Stop Loss   -0.151786        0.004464       -0.151786       -0.151786     -0.155556
   CDN 2023-01-03     23374.96              23374.96 2023-03-31    23549.40 Keep Position    0.007463        0.119403       -0.0223